# Content-Based Recommendation System Preprocessing

In this notebook we conducted preprocessing for the data used in content-based recommendation systems. 

In [1]:
import pandas as pd 
import numpy as np 
import re
import ast

In [2]:
wines = pd.read_csv('../dataset/XWines_Full_100K_wines.csv')

In [3]:
wines.head()

,WineID,WineName,Type,Elaborate,Grapes,Harmonize,ABV,Body,Acidity,Code,Country,RegionID,RegionName,WineryID,WineryName,Website,Vintages
0,100001,Espumante Moscatel,Sparkling,Varietal/100%,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",7.5,Medium-bodied,High,BR,Brazil,1001,Serra Gaúcha,10001,Casa Perini,http://www.vinicolaperini.com.br,"[2020, 2019, 2018, 2017, 2016, 2015, 2014, 201..."
1,100002,Ancellotta,Red,Varietal/100%,['Ancellotta'],"['Beef', 'Barbecue', 'Codfish', 'Pasta', 'Pizz...",12.0,Medium-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10001,Casa Perini,http://www.vinicolaperini.com.br,"[2016, 2015, 2014, 2013, 2012, 2011, 2010, 200..."
2,100003,Cabernet Sauvignon,Red,Varietal/100%,['Cabernet Sauvignon'],"['Beef', 'Lamb', 'Poultry']",12.0,Full-bodied,High,BR,Brazil,1001,Serra Gaúcha,10002,Castellamare,https://www.emporiocastellamare.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."
3,100004,Virtus Moscato,White,Varietal/100%,['Muscat/Moscato'],['Sweet Dessert'],12.0,Medium-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10003,Monte Paschoal,http://www.montepaschoal.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."
4,100005,Maison de Ville Cabernet-Merlot,Red,Assemblage/Bordeaux Red Blend,"['Cabernet Sauvignon', 'Merlot']","['Beef', 'Lamb', 'Game Meat', 'Poultry']",11.0,Full-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10000,Aurora,http://www.vinicolaaurora.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."


In [4]:
wines.info()

<class 'pandas.DataFrame'>
RangeIndex: 100646 entries, 0 to 100645
Data columns (total 17 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   WineID      100646 non-null  int64  
 1   WineName    100646 non-null  str    
 2   Type        100646 non-null  str    
 3   Elaborate   100646 non-null  str    
 4   Grapes      100646 non-null  str    
 5   Harmonize   100646 non-null  str    
 6   ABV         100646 non-null  float64
 7   Body        100646 non-null  str    
 8   Acidity     100646 non-null  str    
 9   Code        100646 non-null  str    
 10  Country     100646 non-null  str    
 11  RegionID    100646 non-null  int64  
 12  RegionName  100646 non-null  str    
 13  WineryID    100646 non-null  int64  
 14  WineryName  100646 non-null  str    
 15  Website     82779 non-null   str    
 16  Vintages    100646 non-null  str    
dtypes: float64(1), int64(3), str(13)
memory usage: 13.1 MB


As we have already established in the exploratory data analysis part of our project, only the Website column contains missing data, which does not present much of a challenge bearing in mind that the website is not something that impacts the recommendations. 

In [5]:
useful_cols = [
    "WineID",
    "WineName",
    "Type",
    "Elaborate",
    "Grapes",
    "Harmonize",
    "ABV",
    "Body",
    "Acidity",
    "Country",
    "RegionName",
    "WineryName"
]

We picked out the columns we are going to use and we found the Winery ID and Region ID redundant as we are using the Winery and Region name, as well as the Code. Vintages can contain some temporal information but it is less meaningful in terms of the sensory and stylistic similarity, so we decided to exclude it. 

In [6]:
wines_cb = wines[useful_cols].copy()

In [7]:
wines_cb.info()

<class 'pandas.DataFrame'>
RangeIndex: 100646 entries, 0 to 100645
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   WineID      100646 non-null  int64  
 1   WineName    100646 non-null  str    
 2   Type        100646 non-null  str    
 3   Elaborate   100646 non-null  str    
 4   Grapes      100646 non-null  str    
 5   Harmonize   100646 non-null  str    
 6   ABV         100646 non-null  float64
 7   Body        100646 non-null  str    
 8   Acidity     100646 non-null  str    
 9   Country     100646 non-null  str    
 10  RegionName  100646 non-null  str    
 11  WineryName  100646 non-null  str    
dtypes: float64(1), int64(1), str(10)
memory usage: 9.2 MB


In [8]:
wines_cb[["Grapes", "Harmonize"]].head()


,Grapes,Harmonize
0,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']"
1,['Ancellotta'],"['Beef', 'Barbecue', 'Codfish', 'Pasta', 'Pizz..."
2,['Cabernet Sauvignon'],"['Beef', 'Lamb', 'Poultry']"
3,['Muscat/Moscato'],['Sweet Dessert']
4,"['Cabernet Sauvignon', 'Merlot']","['Beef', 'Lamb', 'Game Meat', 'Poultry']"


In [9]:
print(type(wines_cb.loc[0, "Grapes"]))

<class 'str'>


In [10]:
import ast

list_cols = ["Grapes", "Harmonize"]

for col in list_cols:
    wines_cb[col] = wines_cb[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [11]:
print(wines_cb.loc[0, "Grapes"])
print(type(wines_cb.loc[0, "Grapes"]))

['Muscat/Moscato']
<class 'list'>


Some of the columns, like Grapes and Harmonize contained data that is list-like and should not be in the string format for the sake of feature engineering, token creation and similarity computation we decided to turn them into lists. 

In [12]:
import re

text_cols = [
    "WineName",
    "Type",
    "Elaborate",
    "Body",
    "Acidity",
    "Country",
    "RegionName",
    "WineryName"
]

list_cols = ["Grapes", "Harmonize"]


def normalize_text_token(text):
    """
    For BoW / TF-IDF style token features:
    south africa -> south_africa
    full-bodied -> full_bodied
    """
    if not isinstance(text, str):
        return ""
    
    text = text.lower().strip()
    text = re.sub(r"[/\-]", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace(" ", "_")
    return text


def normalize_text_natural(text):
    """
    For BERT-style natural language:
    south africa -> south africa
    full-bodied -> full bodied
    """
    if not isinstance(text, str):
        return ""
    
    text = text.lower().strip()
    text = re.sub(r"[/\-]", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_list_token(values):
    if not isinstance(values, list):
        return []
    return [normalize_text_token(v) for v in values if isinstance(v, str) and v.strip()]


def normalize_list_natural(values):
    if not isinstance(values, list):
        return []
    return [normalize_text_natural(v) for v in values if isinstance(v, str) and v.strip()]

In [13]:
# Make a copy so you keep the original selected dataframe safe
wines_features = wines_cb.copy()

# token-style columns for BoW / TF-IDF
for col in text_cols:
    wines_features[f"{col}_tok"] = wines_features[col].apply(normalize_text_token)

for col in list_cols:
    wines_features[f"{col}_tok"] = wines_features[col].apply(normalize_list_token)

# natural-language columns for BERT
for col in text_cols:
    wines_features[f"{col}_nat"] = wines_features[col].apply(normalize_text_natural)

for col in list_cols:
    wines_features[f"{col}_nat"] = wines_features[col].apply(normalize_list_natural)

In [14]:
def bin_abv(abv):
    if pd.isna(abv):
        return "abv_unknown"
    elif abv < 11:
        return "abv_low"
    elif abv < 12:
        return "abv_11_12"
    elif abv < 13:
        return "abv_12_13"
    elif abv < 14:
        return "abv_13_14"
    else:
        return "abv_high"

wines_features["ABV_bin"] = wines_features["ABV"].apply(bin_abv)
wines_features["ABV_nat"] = wines_features["ABV"].round(1).astype(str) + "% abv"

In [15]:
def build_content_text(row):
    tokens = []
    
    # scalar token features
    if row["Type_tok"]:
        tokens.append(f"type_{row['Type_tok']}")
    if row["Elaborate_tok"]:
        tokens.append(f"elaborate_{row['Elaborate_tok']}")
    if row["Body_tok"]:
        tokens.append(f"body_{row['Body_tok']}")
    if row["Acidity_tok"]:
        tokens.append(f"acidity_{row['Acidity_tok']}")
    if row["Country_tok"]:
        tokens.append(f"country_{row['Country_tok']}")
    if row["RegionName_tok"]:
        tokens.append(f"region_{row['RegionName_tok']}")
    if row["WineryName_tok"]:
        tokens.append(f"winery_{row['WineryName_tok']}")
    if row["WineName_tok"]:
        tokens.append(f"wine_{row['WineName_tok']}")
    if row["ABV_bin"]:
        tokens.append(row["ABV_bin"])
    
    # multi-value token features
    tokens.extend([f"grape_{g}" for g in row["Grapes_tok"] if g])
    tokens.extend([f"food_{h}" for h in row["Harmonize_tok"] if h])
    
    return " ".join(tokens)

wines_features["content_text"] = wines_features.apply(build_content_text, axis=1)

In [16]:
def join_list_natural(values):
    if not values:
        return ""
    return ", ".join(values)


def build_bert_text(row):
    parts = []
    
    wine_name = row["WineName_nat"]
    wine_type = row["Type_nat"]
    body = row["Body_nat"]
    acidity = row["Acidity_nat"]
    country = row["Country_nat"]
    region = row["RegionName_nat"]
    elaborate = row["Elaborate_nat"]
    winery = row["WineryName_nat"]
    grapes = join_list_natural(row["Grapes_nat"])
    foods = join_list_natural(row["Harmonize_nat"])
    abv = row["ABV_nat"]
    
    if wine_name:
        parts.append(f"{wine_name}.")
    
    description = "This wine"
    
    descriptors = []
    if body:
        descriptors.append(body)
    if acidity:
        descriptors.append(f"{acidity} acidity")
    if wine_type:
        descriptors.append(wine_type)
    
    if descriptors:
        description += " is a " + ", ".join(descriptors) + " wine"
    else:
        description += " is a wine"
    
    if region and country:
        description += f" from {region}, {country}"
    elif country:
        description += f" from {country}"
    elif region:
        description += f" from {region}"
    
    description += "."
    parts.append(description)
    
    if elaborate:
        parts.append(f"It is classified as {elaborate}.")
    
    if grapes:
        parts.append(f"It is made from {grapes}.")
    
    if foods:
        parts.append(f"It pairs well with {foods}.")
    
    if abv:
        parts.append(f"It has {abv}.")
    
    if winery:
        parts.append(f"The winery is {winery}.")
    
    return " ".join(parts)

wines_features["bert_text"] = wines_features.apply(build_bert_text, axis=1)

In [17]:
wines_features[["WineID", "content_text", "bert_text"]].head(3)

,WineID,content_text,bert_text
0,100001,type_sparkling elaborate_varietal_100 body_med...,espumante moscatel. This wine is a medium bodi...
1,100002,type_red elaborate_varietal_100 body_medium_bo...,"ancellotta. This wine is a medium bodied, medi..."
2,100003,type_red elaborate_varietal_100 body_full_bodi...,cabernet sauvignon. This wine is a full bodied...


In [18]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
wines_features.to_csv("../data/processed/wines_features.csv", index=False)